In [0]:
dbutils.widgets.text("catalog", "dbr_dev_ua5816bd")
dbutils.widgets.text("silver_schema", "viktoriia_kalenichenko_silver")
dbutils.widgets.text("silver_table", "environment")

In [0]:
catalog = dbutils.widgets.get("catalog")
silver_schema = dbutils.widgets.get("silver_schema")
silver_table_name = dbutils.widgets.get("silver_table")

In [0]:
silver_table = f"{catalog}.{silver_schema}.{silver_table_name}"

##Schema Enforcement


In [0]:
from pyspark.sql import functions as F
silver_df = spark.table(silver_table)

In [0]:
wrong_schema_df = (silver_df.limit(10).withColumn("YEAR", F.col("YEAR").cast("string")))

In [0]:
wrong_schema_df.printSchema()

In [0]:
try:
    (
        wrong_schema_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(silver_table)
    )

    print("Unexpected: write succeeded, schema was not enforced")

except Exception as e:
    print("Expected: write rejected -", type(e).__name__)

In [0]:
bad_df = (
    spark.table(silver_table)
    .limit(1)
    .withColumnRenamed("AREA", "unexpected_column")
)

try:
    (
        bad_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(silver_table)
    )

    print("Unexpected: write succeeded, schema was not enforced")

except Exception as e:
    print("Expected: write rejected -", type(e).__name__)

##Schema Evolution

In [0]:
evolution_df = (
    silver_df
    .withColumn(
        "has_missing_metrics",
        F.col("freshwater_per_capita").isNull()
        | F.col("freshwater_withdrawals").isNull()
        | F.col("agricultural_land").isNull()
        | F.col("forest_area").isNull()
    )
)

In [0]:
(evolution_df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(silver_table))
spark.table(silver_table).printSchema()

In [0]:
# Adding a widening data
# Column mapping
spark.sql(f"""
ALTER TABLE {silver_table} SET TBLPROPERTIES (
'delta.enableTypeWidening' = 'true')
""")


In [0]:
widening_df = (spark.table(silver_table).limit(10).withColumn("YEAR", F.col("YEAR").cast("long")))

In [0]:
(
    widening_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(silver_table)
)
spark.table(silver_table).printSchema()

###Delta column mapping: rename and drop columns safely

In [0]:
spark.sql(f"""
ALTER TABLE {silver_table} SET TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name'
)
""")

In [0]:
#rename
spark.sql(f"""
ALTER TABLE {silver_table} RENAME COLUMN has_missing_metrics TO missing_metrics_flag
""")

In [0]:
#drop
spark.sql(f"""
    ALTER TABLE {silver_table} DROP COLUMN missing_metrics_flag
""")

Data contracts define the expected structure and quality of incoming data before it reaches Silver.
Instead of allowing any schema change silently, a contract can specify required columns, data types, nullable rules, and business constraints. Schema evolution should therefore be controlled and intentional.